# 01 — Residential Mortgage LGD Model

**Australian APRA-Style IRB LGD Framework**

This notebook demonstrates a complete residential mortgage LGD workflow aligned with
Australian Big 4 / APRA practice:

| Step | Description |
|------|-------------|
| 1 | Load synthetic mortgage default & workout data |
| 2 | Exploratory data analysis — LGD distributions, cure rates |
| 3 | Segmentation — Standard/Non-standard × LTV band × LMI |
| 4 | Two-stage model — P(Cure) logistic + LGD\|Loss beta regression |
| 5 | Exposure-weighted long-run LGD |
| 6 | Downturn overlay (macro-linked) |
| 7 | Margin of conservatism |
| 8 | APRA overlays — LMI, floors (10%/15%), standard/non-standard |
| 9 | Illustrative RWA and capital linkage |
| 10 | Validation — accuracy, calibration, PSI, out-of-time backtest |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, classification_report
import statsmodels.api as sm

from src.data.data_generation import generate_mortgage_data
from src.lgd_calculation import MortgageLGDEngine, exposure_weighted_average
from src.validation import (
    accuracy_metrics, discriminatory_power, conservatism_test,
    calibration_by_segment, compute_psi, out_of_time_split,
    out_of_time_backtest, generate_validation_report
)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'tables'), exist_ok=True)

## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** proxy arrears and behavioural drivers are allowed for portfolio demonstration where observed collections history is unavailable.
- **Cure treatment (standard policy):** mortgages use two-stage cure logic `LGD = (1 - cure_rate) * liquidation_loss`.
- **Calibration status (standard policy):** this notebook is a transparent proxy baseline and is **not** internally calibrated to real workout tapes.
- **Product differentiation:** mortgage severity is driven by LVR/LMI and the cure-vs-foreclosure path rather than DSCR/ICR or development completion metrics.
- **Output standard:** report `lgd_base`, `lgd_downturn`, and EAD-weighted outputs for interview-ready comparison.


## Objective
Build an interview-ready residential mortgage LGD module with transparent cure-vs-liquidation logic and exposure-weighted outputs.

## Drivers
- LVR at default
- LMI eligibility and coverage
- Arrears/behavioural proxy inputs
- Cure probability and foreclosure path

## Logic
- Two-stage mortgage treatment: `LGD = (1 - cure_rate) * liquidation_loss`
- Product-appropriate discounting and weighted LGD aggregation

## Downturn
- Macro-linked stress via house-price decline, unemployment, and rate shock channels

## Validation
- EAD-weighted base/downturn outputs, stability checks, and governance flags

## Limitations
- Synthetic portfolio and proxy assumptions; future calibration requires internal workout data


## 1. Generate & Load Data

In [ ]:
loans, cashflows = generate_mortgage_data(n_loans=500, seed=42)

print(f"Loans: {loans.shape}")
print(f"Cashflows: {cashflows.shape}")
print(f"\nDefault date range: {loans['default_date'].min()} to {loans['default_date'].max()}")
print(f"Cure rate: {(loans['resolution_type'] == 'Cure').mean():.1%}")
loans.head()

## 2. Exploratory Data Analysis

In [ ]:
# Summary statistics
print("=== Realised LGD Summary ===")
display(loans['realised_lgd'].describe())

print("\n=== By Resolution Type ===")
display(loans.groupby('resolution_type')['realised_lgd'].describe())

In [ ]:
# Bimodal LGD distribution - the key insight driving two-stage modelling
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ewa_realised = exposure_weighted_average(loans, 'realised_lgd', 'ead')
axes[0].hist(loans['realised_lgd'], bins=40, edgecolor='white', alpha=0.8)
axes[0].set_title('All Defaults - Realised LGD')
axes[0].set_xlabel('LGD')
axes[0].axvline(ewa_realised, color='red', ls='--', label=f"EAD-weighted: {ewa_realised:.2%}")
axes[0].legend()

cures = loans[loans['resolution_type'] == 'Cure']['realised_lgd']
losses = loans[loans['resolution_type'] == 'Property Sale']['realised_lgd']

axes[1].hist(cures, bins=30, edgecolor='white', alpha=0.8, color='green')
axes[1].set_title(f'Cures (n={len(cures)}) - Near-Zero LGD')
axes[1].set_xlabel('LGD')

axes[2].hist(losses, bins=30, edgecolor='white', alpha=0.8, color='salmon')
axes[2].set_title(f'Property Sales (n={len(losses)}) - Loss Cases')
axes[2].set_xlabel('LGD')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_lgd_bimodal.png'), dpi=150, bbox_inches='tight')
plt.show()

print()
print('Bimodal distribution confirms need for TWO-STAGE model (cure vs loss).')


In [ ]:
# LTV at default vs LGD — the primary risk driver
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Colour by resolution
for res, colour in [('Cure', 'green'), ('Property Sale', 'salmon')]:
    mask = loans['resolution_type'] == res
    axes[0].scatter(loans.loc[mask, 'ltv_at_default'], loans.loc[mask, 'realised_lgd'],
                    alpha=0.4, s=15, label=res, color=colour)
axes[0].set_xlabel('LTV at Default')
axes[0].set_ylabel('Realised LGD')
axes[0].set_title('LTV at Default vs Realised LGD')
axes[0].legend()

# Boxplot by LTV band
engine = MortgageLGDEngine()
segmented = engine.segment_loans(loans)
segmented.boxplot(column='realised_lgd', by='ltv_band', ax=axes[1])
axes[1].set_title('Realised LGD by LTV Band')
axes[1].set_xlabel('LTV Band')
axes[1].set_ylabel('Realised LGD')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_ltv_vs_lgd.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Key driver analysis
print('=== Exposure-Weighted LGD by Key Segments ===')
for col in ['mortgage_class', 'property_type', 'occupancy', 'loan_type', 'state']:
    print()
    print(f"--- {col} ---")
    rows = []
    for seg, grp in loans.groupby(col, observed=True):
        rows.append({
            col: seg,
            'count': len(grp),
            'total_ead': grp['ead'].sum(),
            'cure_rate': (grp['resolution_type'] == 'Cure').mean(),
            'ead_weighted_lgd': exposure_weighted_average(grp, 'realised_lgd', 'ead'),
            'ead_weighted_ltv': exposure_weighted_average(grp, 'ltv_at_default', 'ead'),
        })
    summary = pd.DataFrame(rows).set_index(col).sort_values('ead_weighted_lgd', ascending=False).round(4)
    display(summary)


## 3. Segmentation

Australian mortgage LGD segmentation hierarchy:
```
Level 1: Standard vs Non-Standard  (APRA requirement)
  Level 2: LTV band at default     (primary risk driver)
    Level 3: LMI eligible           (material recovery impact)
```

In [ ]:
# Exposure-weighted long-run LGD by primary segments
engine = MortgageLGDEngine()
lr_lgd = engine.compute_long_run_lgd(loans, segments=['mortgage_class', 'ltv_band'])

print("=== Exposure-Weighted Long-Run LGD ===")
display(lr_lgd)

# Pivot for heatmap
pivot = lr_lgd.pivot(index='mortgage_class', columns='ltv_band', values='lgd_long_run')
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='YlOrRd', ax=ax)
ax.set_title('Exposure-Weighted Long-Run LGD: Mortgage Class × LTV Band')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_segment_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Cure Treatment (Pre-Step-3 Baseline)

Cure modelling is mandatory in the mortgage baseline because many defaults resolve before liquidation.

- Stage 1: estimate `P(cure)` from approved proxy drivers.
- Stage 2: estimate liquidation loss conditional on non-cure.
- Combine: `LGD = (1 - cure_rate) * liquidation_loss`.

Policy position: fallback and proxy rules follow `docs/fallback_hierarchy.md`; calibration remains proxy-only pending internal data.


In [ ]:
# Prepare modelling data with transparent cure proxies
model_df = loans.copy()
model_df['is_cure'] = (model_df['resolution_type'] == 'Cure').astype(int)

# Borrower type proxy
model_df['borrower_type_proxy'] = np.where(
    model_df['occupancy'] == 'Investor',
    'Investor',
    'Owner-Occupier',
)
model_df['is_investor'] = (model_df['borrower_type_proxy'] == 'Investor').astype(int)

# Arrears stage proxy at default (30-59 / 60-89 / 90+ DPD style buckets)
model_df['arrears_stage_proxy'] = np.select(
    [
        (model_df['ltv_at_default'] >= 0.95) | (model_df['dti'] >= 0.45) | (model_df['credit_score'] < 560),
        (model_df['ltv_at_default'] >= 0.85) | (model_df['dti'] >= 0.35) | (model_df['credit_score'] < 640),
    ],
    ['90+ DPD (Proxy)', '60-89 DPD (Proxy)'],
    default='30-59 DPD (Proxy)',
)
arrears_map = {
    '30-59 DPD (Proxy)': 0,
    '60-89 DPD (Proxy)': 1,
    '90+ DPD (Proxy)': 2,
}
model_df['arrears_stage_num'] = model_df['arrears_stage_proxy'].map(arrears_map).astype(int)

# Repayment behaviour proxy (simple score from borrower risk profile)
repay_score = (
    (model_df['loan_type'] == 'P&I').astype(int)
    + (model_df['credit_score'] >= 700).astype(int)
    + (model_df['seasoning_months'] >= 24).astype(int)
    + (model_df['dti'] <= 0.35).astype(int)
    - (model_df['ltv_at_default'] > 0.90).astype(int)
)
model_df['repayment_behaviour_proxy'] = np.select(
    [repay_score >= 3, repay_score >= 1],
    ['Strong', 'Stable'],
    default='Weak',
)
repayment_map = {'Weak': 0, 'Stable': 1, 'Strong': 2}
model_df['repayment_behaviour_num'] = model_df['repayment_behaviour_proxy'].map(repayment_map).astype(int)

# Additional standard risk flags
model_df['is_non_standard'] = (model_df['mortgage_class'] == 'Non-Standard').astype(int)

features = [
    'ltv_at_default',
    'arrears_stage_num',
    'is_investor',
    'repayment_behaviour_num',
    'is_non_standard',
    'lmi_flag',
]

print('=== Cure Proxy Distributions ===')
display(model_df['arrears_stage_proxy'].value_counts(dropna=False).rename('count').to_frame())
display(model_df['repayment_behaviour_proxy'].value_counts(dropna=False).rename('count').to_frame())


In [ ]:
# STAGE 1: Cure rate model (logistic)
X = model_df[features]
y_cure = model_df['is_cure']

cure_model = LogisticRegression(max_iter=2000, random_state=42)
cure_cv_proba = cross_val_predict(cure_model, X, y_cure, cv=5, method='predict_proba')[:, 1]
cure_model.fit(X, y_cure)

cure_auc = roc_auc_score(y_cure, cure_cv_proba)
print(f"Stage 1 - Cure Model AUC (5-fold CV): {cure_auc:.4f}")

cure_coefs = pd.DataFrame(
    {
        'Feature': features,
        'Coefficient': cure_model.coef_[0],
    }
).sort_values('Coefficient', ascending=False)
print()
print('Cure Model Coefficients:')
display(cure_coefs.round(4))

model_df['cure_rate'] = cure_model.predict_proba(X)[:, 1]


In [ ]:
# STAGE 2: Liquidation loss model (non-cure loans only)
loss_df = model_df[model_df['is_cure'] == 0].copy()
loss_df['liquidation_loss_observed'] = loss_df['realised_lgd']

# Logit transform for bounded target
loss_df['liquidation_loss_clipped'] = loss_df['liquidation_loss_observed'].clip(0.001, 0.999)
loss_df['liquidation_loss_logit'] = np.log(
    loss_df['liquidation_loss_clipped'] / (1 - loss_df['liquidation_loss_clipped'])
)

X_loss = sm.add_constant(loss_df[features])
y_loss = loss_df['liquidation_loss_logit']
loss_model = sm.OLS(y_loss, X_loss).fit()

print('Stage 2 - Liquidation Loss Model (non-cure loans, logit OLS)')
print(f"R-squared: {loss_model.rsquared:.4f}")
print(f"Adj. R-squared: {loss_model.rsquared_adj:.4f}")

loss_coefs = pd.DataFrame(
    {
        'Feature': ['const'] + features,
        'Coefficient': loss_model.params,
        'P-value': loss_model.pvalues,
    }
)
display(loss_coefs.round(4))


In [ ]:
# COMBINE STAGES: LGD = (1 - cure_rate) * liquidation_loss
X_all = sm.add_constant(model_df[features])
liquidation_logit_pred = loss_model.predict(X_all)
model_df['liquidation_loss'] = (1 / (1 + np.exp(-liquidation_logit_pred))).clip(0, 1)

model_df['lgd_predicted'] = (1 - model_df['cure_rate']) * model_df['liquidation_loss']

print('=== Two-Stage Cure Model Output ===')
print(f"EAD-weighted Cure Rate:        {exposure_weighted_average(model_df, 'cure_rate', 'ead'):.2%}")
print(f"EAD-weighted Liquidation Loss: {exposure_weighted_average(model_df, 'liquidation_loss', 'ead'):.2%}")
print(f"EAD-weighted Predicted LGD:    {exposure_weighted_average(model_df, 'lgd_predicted', 'ead'):.2%}")
print(f"EAD-weighted Actual LGD:       {exposure_weighted_average(model_df, 'realised_lgd', 'ead'):.2%}")

cure_segment = (
    model_df.groupby(['arrears_stage_proxy', 'repayment_behaviour_proxy'], observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_weighted_cure_rate': exposure_weighted_average(g, 'cure_rate', 'ead'),
                'ead_weighted_lgd_predicted': exposure_weighted_average(g, 'lgd_predicted', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
print()
print('=== Cure Segment Diagnostics ===')
display(cure_segment.round(4))


In [ ]:
# Predicted vs actual scatter
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(model_df['realised_lgd'], model_df['lgd_predicted'], alpha=0.3, s=15)
ax.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Perfect calibration')
ax.set_xlabel('Actual Realised LGD')
ax.set_ylabel('Predicted LGD (Two-Stage)')
ax.set_title('Two-Stage Model: Predicted vs Actual')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_pred_vs_actual.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5–8. Regulatory Pipeline: Long-Run → Downturn → MoC → APRA Overlays

In [ ]:
# Use two-stage cure model LGD as input to regulatory overlays
loans_for_reg = model_df.copy()
loans_for_reg['realised_lgd_actual'] = loans_for_reg['realised_lgd']
loans_for_reg['realised_lgd'] = loans_for_reg['lgd_predicted']

engine = MortgageLGDEngine(downturn_scalar=1.08, moc=0.02)
result = engine.apply_apra_overlays(loans_for_reg)

print('=== Regulatory Pipeline Summary ===')
pipeline_cols = ['lgd_base', 'lgd_downturn', 'lgd_with_moc', 'lgd_after_lmi', 'lgd_final']
display(result[pipeline_cols].describe().round(4))

print()
print('=== Exposure-Weighted Portfolio Averages ===')
for col in pipeline_cols:
    ewa = exposure_weighted_average(result, lgd_col=col, ead_col='ead')
    print(f"  {col:25s}: {ewa:.4%}")

print()
print('=== Downturn Scalar (Mortgage Macro-Linked) ===')
print(
    f"min={result['downturn_scalar'].min():.3f}, "
    f"mean={result['downturn_scalar'].mean():.3f}, "
    f"max={result['downturn_scalar'].max():.3f}"
)


In [ ]:
# Waterfall: Two-Stage LGD -> Downturn -> MoC -> LMI -> Final
pipeline_means = {col: exposure_weighted_average(result, col) for col in pipeline_cols}

fig, ax = plt.subplots(figsize=(10, 5))
labels = ['Base LGD (Cure-Aware)', 'Downturn (Macro)', '+ MoC', 'LMI Adjustment', 'Final (Floor Applied)']
values = [pipeline_means[c] for c in pipeline_cols]
colors = ['#3498db', '#e67e22', '#e74c3c', '#2ecc71', '#8e44ad']

bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{val:.2%}',
        ha='center',
        va='bottom',
        fontweight='bold',
    )

ax.set_ylabel('Exposure-Weighted LGD')
ax.set_title('Mortgage LGD Pipeline with Cure Modelling')
ax.set_ylim(0, max(values) * 1.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'mortgage_waterfall.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# LGD by mortgage class - Standard vs Non-Standard floor impact
for mc in ['Standard', 'Non-Standard']:
    subset = result[result['mortgage_class'] == mc]
    floor = 0.10 if mc == 'Standard' else 0.15
    floored_pct = ((subset['lgd_after_lmi'] < floor).sum() / len(subset)) if len(subset) else 0.0
    input_ewa = exposure_weighted_average(subset, 'realised_lgd', 'ead')
    final_ewa = exposure_weighted_average(subset, 'lgd_final', 'ead')
    print()
    print(f"{mc} Mortgages (n={len(subset)}):")
    print(f"  EAD-weighted Input LGD:  {input_ewa:.2%}")
    print(f"  EAD-weighted Final LGD:  {final_ewa:.2%}")
    print(f"  Floor ({floor:.0%}) binding: {floored_pct:.1%} of loans")


## 9. Illustrative RWA & Capital

In [ ]:
result_rwa = engine.compute_illustrative_rwa(result, pd_estimate=0.02)

print("=== Illustrative Capital Metrics ===")
print(f"Total EAD:              ${result_rwa['ead'].sum():>15,.0f}")
print(f"Total RWA (pre-APRA):   ${result_rwa['rwa'].sum():>15,.0f}")
print(f"Total RWA (post-APRA):  ${result_rwa['rwa_after_apra_scalar'].sum():>15,.0f}")
print(f"APRA Scalar Impact:     +{(result_rwa['rwa_after_apra_scalar'].sum() / result_rwa['rwa'].sum() - 1):.1%}")
print(f"Average Risk Weight:    {(result_rwa['rwa_after_apra_scalar'].sum() / result_rwa['ead'].sum()):.2%}")

## 10. Validation & Backtesting

In [ ]:
# Full validation report (actual realised LGD vs final model LGD)
segmented_result = engine.segment_loans(result)
val_report = generate_validation_report(
    segmented_result,
    actual_col='realised_lgd_actual',
    predicted_col='lgd_final',
    segment_col='ltv_band',
    date_col='default_date'
)

print('=== Accuracy ===')
for k, v in val_report['accuracy'].items():
    print(f"  {k}: {v}")

print()
print('=== Discriminatory Power ===')
for k, v in val_report['discriminatory_power'].items():
    print(f"  {k}: {v}")

print()
print('=== Conservatism Test ===')
for k, v in val_report['conservatism'].items():
    print(f"  {k}: {v}")


In [ ]:
# Calibration by LTV band
print("=== Calibration by LTV Band ===")
display(val_report.get('calibration', 'No calibration available'))

In [ ]:
# PSI stability
if 'stability_psi' in val_report:
    psi_result = val_report['stability_psi']
    print(f"PSI: {psi_result['PSI']:.4f} — {psi_result['Interpretation']}")
    display(psi_result['Detail'])

In [ ]:
# Out-of-time backtest
if 'out_of_time' in val_report:
    oot = val_report['out_of_time']
    print(f"Train: {oot['train_size']} loans | Test: {oot['test_size']} loans")
    print(f"\nOut-of-Time Accuracy:")
    for k, v in oot['accuracy'].items():
        print(f"  {k}: {v}")
    print(f"\nOut-of-Time Conservatism:")
    for k, v in oot['conservatism'].items():
        print(f"  {k}: {v}")

In [ ]:
# Save outputs
result_rwa.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mortgage_lgd_results.csv'), index=False)
lr_lgd.to_csv(os.path.join(OUTPUT_DIR, 'tables', 'mortgage_segment_summary.csv'), index=False)

print("Outputs saved to outputs/tables/")

---

## APS 113 Calibration Layer — Residential Mortgage

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section adds a full APS 113-aligned calibration loop on top of the
> proxy baseline above. Workout data is synthetically generated (2014-2024,
> 10-year window) to demonstrate methodology; real production calibration
> requires an internal workout tape.

### Calibration Pipeline (APS 113 s.32-68)

| Step | Function | APS 113 |
|------|----------|---------|
| 1 | Load/generate historical workout data | s.44, Att A |
| 2 | `compute_realised_lgd()` — LIP costs, cure leg | s.32-34 |
| 3 | `classify_economic_regime()` + `assign_regime_to_workouts()` | s.43, s.46 |
| 4 | `segment_lgd()` — product-specific segment keys | s.52 |
| 5 | `compute_long_run_lgd()` — vintage-EWA method | s.43-44 |
| 6 | `compare_model_vs_actual()` — proxy vs realised | s.60-62 |
| 7 | `apply_downturn_overlay()` + `apply_correlation_adjustment()` | s.46-50, s.55-57 |
| 8 | `MoCRegister` + `apply_moc()` — 5 APS 113 s.65 sources | s.63-65 |
| 9 | `apply_regulatory_floor()` — 10% (Standard+LMI) / 15% (Non-Standard) per APS 113 s.58 | s.58 |
| 10 | Export 9 CSV outputs | — |
| 11 | `run_full_validation_suite()` — Gini, HL, PSI, OOT | s.66-68 |

**Correct APS 113 order:** LR-LGD → downturn overlay → correlation adj →
MoC → floor (MoC applied to downturn LGD, not long-run LGD, per s.63).


In [ ]:
# APS 113 Calibration Layer — imports and configuration
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

from src.calibration_utils import (
    compute_realised_lgd,
    segment_lgd,
    compute_long_run_lgd,
    compare_model_vs_actual,
    classify_economic_regime,
    assign_regime_to_workouts,
    apply_downturn_overlay,
    apply_correlation_adjustment,
    build_lgd_pd_annual_series,
    estimate_lgd_pd_correlation,
    MoCRegister,
    apply_moc,
    apply_regulatory_floor,
    run_full_validation_suite,
    generate_compliance_map,
    export_compliance_map,
    CALIBRATION_STEP_ORDER,
)
from src.generators import GENERATOR_MAP, generate_all_historical_workouts

PRODUCT = "mortgage"
SEGMENT_KEYS = ['mortgage_class', 'lvr_band']
MODEL_LGD_COL = "lgd_final"

HISTORY_DIR = Path('..') / 'data' / 'generated' / 'historical'
OUTPUTS_DIR = Path('..') / 'outputs' / 'tables'
UPSTREAM_PARQUET = Path('..') / 'data' / 'exports' / 'macro_regime_flags.parquet'

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Product: {PRODUCT}")
print(f"Segment keys: {SEGMENT_KEYS}")
print(f"APS 113 calibration pipeline — step order:")
for step, fn, ref in CALIBRATION_STEP_ORDER:
    print(f"  Step {step:>2}: {fn:<35} {ref}")


In [ ]:
# ── STEP 1: Load or generate historical workout data (APS 113 s.44, Att A) ─
parquet_path = HISTORY_DIR / f"{PRODUCT}_workouts.parquet"

if parquet_path.exists():
    cal_loans = pd.read_parquet(parquet_path)
    # cashflows stored alongside or re-generated
    try:
        cal_cashflows = pd.read_parquet(
            HISTORY_DIR / f"{PRODUCT}_cashflows.parquet"
        )
    except FileNotFoundError:
        cal_cashflows = None
    print(f"Loaded {len(cal_loans):,} historical workout loans from Parquet")
else:
    print(f"Parquet not found — generating synthetic workout data for {PRODUCT}")
    results = generate_all_historical_workouts(
        seed=42, output_dir=HISTORY_DIR, write_parquet=True, products=[PRODUCT]
    )
    cal_loans = results[PRODUCT]["loans"]
    cal_cashflows = results[PRODUCT].get("cashflows")
    print(f"Generated {len(cal_loans):,} synthetic workout loans (2014-2024)")

print(f"Date range: {cal_loans['default_date'].min()} to {cal_loans['default_date'].max()}")
print(f"Columns: {list(cal_loans.columns)}")

# ── STEP 2: Compute realised LGD (APS 113 s.32-34) ─────────────────────────
# LIP costs (Loss Identification Period, ~90 days) auto-detected if cashflows available
lgd_df = compute_realised_lgd(
    loans=cal_loans,
    cashflows=cal_cashflows,
    ead_col="ead_at_default",
    recovery_col="recovery_amount",
    cost_col="direct_costs",
    lip_window_days=90,
)
print(f"\nStep 2: Realised LGD computed")
print(f"  EAD-weighted realised LGD: {(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum():.2%}")
display(lgd_df[['realised_lgd', 'ead_at_default']].describe().round(4))

# ── STEP 3: Economic regime classification (APS 113 s.43, s.46-50) ─────────
upstream_path = str(UPSTREAM_PARQUET) if UPSTREAM_PARQUET.exists() else None
regime_df = classify_economic_regime(
    upstream_parquet_path=upstream_path,
    method="upstream_first",
)
print(f"\nStep 3: Economic regimes classified")
print(f"  Data source: {regime_df['data_source'].iloc[0]}")
display(regime_df[['year', 'regime', 'is_downturn_period', 'data_source']].head(12))

lgd_df = assign_regime_to_workouts(lgd_df, regime_df)
downturn_pct = lgd_df.get('is_downturn_period', pd.Series([False])).mean()
print(f"  Downturn observations: {downturn_pct:.1%} of portfolio")

# ── STEP 4: Segment by product-specific keys (APS 113 s.52) ────────────────
segmented_df = segment_lgd(lgd_df, SEGMENT_KEYS)
low_count = segmented_df[segmented_df.get('segment_flag', '') == 'low_count'] if 'segment_flag' in segmented_df.columns else pd.DataFrame()
print(f"\nStep 4: Segmentation complete")
print(f"  Segments: {segmented_df.groupby(SEGMENT_KEYS, observed=True).ngroups}")
if not low_count.empty:
    print(f"  Low-count segments flagged (<20 obs): {len(low_count)}")

# ── STEP 5: Long-run LGD — vintage-EWA (APS 113 s.43-44) ─────────────────
lr_lgd_df = compute_long_run_lgd(
    segmented_df,
    segment_keys=SEGMENT_KEYS,
    method="vintage_ewa",
)
print(f"\nStep 5: Long-run LGD by segment (vintage-EWA)")
display(lr_lgd_df.round(4))
lr_lgd_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_long_run_lgd_by_segment.csv", index=False)

# ── STEP 6: Compare model vs actual (APS 113 s.60-62) ──────────────────────
# 'model_lgd' here = proxy model LGD from the section above (lgd_final if present)
if MODEL_LGD_COL in cal_loans.columns:
    compare_input = lgd_df.merge(
        cal_loans[['loan_id', MODEL_LGD_COL] if 'loan_id' in cal_loans.columns else [MODEL_LGD_COL]],
        left_index=True, right_index=True, how='left',
    ) if MODEL_LGD_COL not in lgd_df.columns else lgd_df
else:
    compare_input = lgd_df.copy()
    compare_input['model_lgd'] = lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else 0.25

model_col_to_use = MODEL_LGD_COL if MODEL_LGD_COL in compare_input.columns else 'model_lgd'
mva_df = compare_model_vs_actual(
    compare_input,
    model_lgd_col=model_col_to_use,
    segment_keys=SEGMENT_KEYS,
)
print(f"\nStep 6: Model vs actual comparison")
display(mva_df.round(4))
mva_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_model_vs_actual_comparison.csv", index=False)

# ── STEP 7: Downturn overlay + Frye-Jacobs correlation adj (s.46-50, s.55-57)
# Reuses apply_downturn_overlay from existing lgd_calculation.py
dt_lgd = apply_downturn_overlay(segmented_df, product=PRODUCT)
print(f"\nStep 7: Downturn overlay applied")
if 'lgd_downturn' in dt_lgd.columns:
    ewa_dt = (dt_lgd['lgd_downturn'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    ewa_lr = (dt_lgd['realised_lgd'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    print(f"  EWA Long-run LGD:  {ewa_lr:.2%}")
    print(f"  EWA Downturn LGD:  {ewa_dt:.2%}")
    downturn_col = 'lgd_downturn'
else:
    dt_lgd['lgd_downturn'] = dt_lgd['realised_lgd'] * 1.15  # fallback scalar
    downturn_col = 'lgd_downturn'

# Frye-Jacobs correlation adjustment (APS 113 s.55-57)
try:
    lgd_ts, pd_ts = build_lgd_pd_annual_series(dt_lgd)
    macro_for_corr = regime_df.rename(columns={'gdp_growth': 'gdp_growth_yoy'})
    corr_result = estimate_lgd_pd_correlation(lgd_ts, pd_ts, macro_for_corr)
    dt_lgd['lgd_downturn_corr_adj'] = apply_correlation_adjustment(
        dt_lgd[downturn_col], corr_result['rho'], corr_result['macro_shock_std']
    )
    downturn_col = 'lgd_downturn_corr_adj'
    print(f"  Frye-Jacobs rho={corr_result['rho']:.3f}, adj factor={corr_result['lgd_dt_adjustment_factor']:.3f}")
except Exception as exc:
    print(f"  Frye-Jacobs skipped: {exc}")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_downturn_lgd_by_segment.csv", index=False)

# ── STEP 8: MoC register + apply MoC (AFTER downturn — APS 113 s.63-65) ───
# Determine regime data source for MoC model_approximation component
data_src = regime_df['data_source'].iloc[0] if 'data_source' in regime_df.columns else 'synthetic'
n_downturn_yrs = int(regime_df['is_downturn_period'].sum()) if 'is_downturn_period' in regime_df.columns else 0

psi_approx = 0.05  # placeholder — full PSI computed in Step 11
bias_approx = float(mva_df['bias'].abs().mean()) if 'bias' in mva_df.columns else 0.02

moc_register = MoCRegister(product=PRODUCT, regime_data_source=data_src)
moc_df = moc_register.build_moc_register(
    segment_df=segmented_df,
    segment_keys=SEGMENT_KEYS,
    n_downturn_vintages=n_downturn_yrs,
    psi_value=psi_approx,
    backtesting_bias=bias_approx,
)
print(f"\nStep 8: MoC register built")
display(moc_df.round(4))
moc_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_moc_register.csv", index=False)

lgd_with_moc = apply_moc(dt_lgd[downturn_col], moc_df, segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None)
dt_lgd['lgd_with_moc'] = lgd_with_moc
moc_ewa = (lgd_with_moc * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
print(f"  EWA LGD after MoC: {moc_ewa:.2%}")

# ── STEP 9: Regulatory floors (APS 113 s.58) ──────────────────────────────
dt_lgd['lgd_final_calibrated'] = apply_regulatory_floor(dt_lgd['lgd_with_moc'], product=PRODUCT)
final_ewa = (dt_lgd['lgd_final_calibrated'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
floor_binding_pct = (dt_lgd['lgd_final_calibrated'] > dt_lgd['lgd_with_moc']).mean()
print(f"\nStep 9: Regulatory floor applied")
print(f"  EWA Final Calibrated LGD: {final_ewa:.2%}")
print(f"  Floor binding for: {floor_binding_pct:.1%} of loans")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_final_calibrated_lgd.csv", index=False)

# ── STEP 10: Export all outputs ────────────────────────────────────────────
# Already exported: long_run_lgd_by_segment, model_vs_actual, downturn_lgd, moc_register, final_calibrated_lgd
# Export remaining:
lgd_df[['realised_lgd', 'ead_at_default']].assign(product=PRODUCT).to_csv(
    OUTPUTS_DIR / f"{PRODUCT}_historical_workouts.csv", index=False
)
regime_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_regime_classification.csv", index=False)

# Calibration adjustments summary
cal_adj_summary = pd.DataFrame({
    'product': [PRODUCT],
    'ewa_realised_lgd': [(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum()],
    'ewa_long_run_lgd': [lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None],
    'ewa_downturn_lgd': [(dt_lgd.get('lgd_downturn', dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_with_moc': [(dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_final': [final_ewa],
    'floor_binding_pct': [floor_binding_pct],
    'regime_data_source': [data_src],
    'n_loans': [len(lgd_df)],
    'calibration_date': [pd.Timestamp.today().date()],
})
cal_adj_summary.to_csv(OUTPUTS_DIR / f"{PRODUCT}_calibration_adjustments.csv", index=False)
print(f"\nStep 10: All outputs exported to {OUTPUTS_DIR}")

# ── STEP 11: Full validation suite (APS 113 s.66-68) ──────────────────────
print(f"\nStep 11: Running full validation suite...")
try:
    val_results = run_full_validation_suite(
        loans=dt_lgd,
        predicted_col='lgd_final_calibrated',
        actual_col='realised_lgd',
        segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None,
        date_col='default_date' if 'default_date' in dt_lgd.columns else None,
        product=PRODUCT,
    )
    print(f"  Gini: {val_results.get('gini', 'n/a')}")
    print(f"  Calibration ratio: {val_results.get('calibration_ratio', 'n/a')}")
    print(f"  PSI: {val_results.get('psi', 'n/a')}")
    if 'summary_table' in val_results:
        val_results['summary_table'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_validation_report.csv", index=False
        )
    if 'backtest_results' in val_results:
        val_results['backtest_results'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_backtest_results.csv", index=False
        )
except Exception as exc:
    print(f"  Validation suite error (non-fatal): {exc}")


In [ ]:
# ── Calibration summary waterfall ──────────────────────────────────────────
import matplotlib.pyplot as plt

try:
    stages = {
        'Realised LGD\n(2014-2024)': (lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum(),
        'Long-run LGD\n(vintage-EWA)': lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None,
        'Downturn LGD': (dt_lgd.get(downturn_col, dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        '+ MoC': (dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        'Final\n(Floor Applied)': final_ewa,
    }
    labels = [k for k, v in stages.items() if v is not None]
    values = [v for v in stages.values() if v is not None]
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad'][:len(labels)]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('EAD-Weighted LGD')
    ax.set_title(f'APS 113 Calibration Waterfall — Residential Mortgage')
    ax.set_ylim(0, max(values) * 1.35)
    ax.axhline(values[-1], color='black', ls=':', lw=1, label=f'Final: {values[-1]:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        Path('..') / 'outputs' / 'figures' / f'mortgage_calibration_waterfall.png',
        dpi=150, bbox_inches='tight',
    )
    plt.show()
except Exception as exc:
    print(f"Waterfall chart error (non-fatal): {exc}")

# ── APS 113 compliance snapshot ─────────────────────────────────────────────
compliance_df = generate_compliance_map(
    calibration_results={PRODUCT: {'long_run_lgd_by_segment': True, 'calibration_steps': True}},
    moc_registers={PRODUCT: moc_df},
    regime_data_source=data_src,
    products=[PRODUCT],
)
print("\n=== APS 113 Compliance Snapshot ===")
display(compliance_df[['section_ref', 'requirement', 'status', 'reviewer_note']].set_index('section_ref'))
export_compliance_map(compliance_df, OUTPUTS_DIR / f"mortgage_aps113_compliance.csv")

# Final summary
print("\n=== Calibration Summary ===")
display(cal_adj_summary.round(4))
print(f"\nAll calibration outputs in: {OUTPUTS_DIR}")
print(f"SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY")
